In [1]:
import numpy as np
import torch
import json
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.transforms import ToTensor
from handToDigitsCNN import HandToDigitsCNN
from croppedImageDataSet import CroppedImageDataSet

In [9]:
MODEL_PATH = "handToDigits_model.pth"
IMAGES_DIR = 'data/CROPPED_IMAGES'
ANNOTATIONS_DIR = 'data/CROPPED_IMAGES/cropped_annotations.json'

transform = transforms.Compose([
    transforms.ToTensor()
])

cropped_image_train_dataset = CroppedImageDataSet(image_dir=IMAGES_DIR,
                                      annotation_dir=ANNOTATIONS_DIR,
                                      transform=transform)

cropped_image_test_dataset = CroppedImageDataSet(image_dir=IMAGES_DIR,
                                      annotation_dir=ANNOTATIONS_DIR,
                                      transform=transform,
                                      train=False)

BATCH_SIZE = 128

trainDataLoader = DataLoader(cropped_image_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
testDataLoader = DataLoader(cropped_image_test_dataset, batch_size=BATCH_SIZE)
for X, y in trainDataLoader:
    print(f"Shape of X: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

device = ("cuda"
          if torch.cuda.is_available()
          else "mps"
          if torch.backends.mps.is_available()
          else "cpu"
         )
print(f"Using {device} as device")

Shape of X: torch.Size([128, 3, 256, 128])
Shape of y: torch.Size([128]) torch.float32
Using mps as device


In [10]:
model = HandToDigitsCNN().to(device)
print(model)

HandToDigitsCNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (convolutional_relu_stack): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
  

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
loss_fn = nn.CrossEntropyLoss()

In [12]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)

        pred = model(x)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (batch) % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(x)
            print(f"Loss: {loss:>7f} [{current:>5d}/{size:>5d}]")
            #print(f"Current pred: {pred.tolist()}")
            #print(f"Current real: {y.tolist()}")

In [13]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()

    average_loss = test_loss / num_batches
    accuracy = correct / size
    print(f"Accuracy: {accuracy}")
    print(f"Average Loss: {average_loss}")
    return accuracy, average_loss

In [14]:
def train_epochs(epochs, given_model, loss_fn, optimizer, forgiveness=1, minimum_epochs=5):
    losses = []
    accuracies = []
    average_loss_min = float('inf')
    misses = 0
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}\n------------------")
        train(trainDataLoader, given_model, loss_fn, optimizer)
        accuracy, average_loss = test(testDataLoader, given_model, loss_fn)
        losses.append(average_loss)
        accuracies.append(accuracy)
        print(f"Epoch Average Loss: {average_loss}, Min Average loss: {average_loss_min}")
        print(f"Is Epoch Avg < Min Avg?: {average_loss < average_loss_min}")
        if average_loss < average_loss_min:
            average_loss_min = average_loss
            torch.save(given_model.state_dict(), "mid_epoch_box_model.pth")
            misses = 0
        elif epoch > minimum_epochs:
            print(f"Loss increased after {epoch+1} epochs")
            misses += 1
            if misses > forgiveness:
                print("miss!")
                given_model = ImageToHandBoxCNN().to(device)
                given_model.load_state_dict(torch.load("mid_epoch_box_model.pth", weights_only=False))
                break
    return losses, accuracies

In [15]:
losses, accuracies = train_epochs(20, model, loss_fn, optimizer)
torch.save(model.state_dict(), MODEL_PATH)

Epoch 1
------------------
Loss: 1.812211 [  128/ 2150]
Accuracy: 0.1691449814126394
Average Loss: 1.7655771017074584
Epoch Average Loss: 1.7655771017074584, Min Average loss: inf
Is Epoch Avg < Min Avg?: True
Epoch 2
------------------
Loss: 0.287596 [  128/ 2150]
Accuracy: 0.5055762081784386
Average Loss: 1.248507571220398
Epoch Average Loss: 1.248507571220398, Min Average loss: 1.7655771017074584
Is Epoch Avg < Min Avg?: True
Epoch 3
------------------
Loss: 0.044218 [  128/ 2150]
Accuracy: 0.9869888475836431
Average Loss: 0.22563355267047883
Epoch Average Loss: 0.22563355267047883, Min Average loss: 1.248507571220398
Is Epoch Avg < Min Avg?: True
Epoch 4
------------------
Loss: 0.015478 [  128/ 2150]
Accuracy: 1.0
Average Loss: 0.025029972195625305
Epoch Average Loss: 0.025029972195625305, Min Average loss: 0.22563355267047883
Is Epoch Avg < Min Avg?: True
Epoch 5
------------------
Loss: 0.008911 [  128/ 2150]
Accuracy: 1.0
Average Loss: 0.0061201279982924465
Epoch Average Loss: 

In [16]:
torch.save(model.state_dict(), MODEL_PATH)